In [ ]:
! pip install torch torchaudio matplotlib numpy scipy inflect unidecode

In [ ]:
import glob
import math
import os
import sys
import pathlib
import pickle
import io

# Repo root whether the notebook cwd is <repo>/tacotron2 or <repo>/
_cwd = os.getcwd()
if os.path.basename(_cwd.rstrip("/\\")) == "tacotron2":
    _REPO_ROOT = os.path.dirname(_cwd)
else:
    _REPO_ROOT = _cwd
for _p in (_REPO_ROOT, os.path.join(_REPO_ROOT, "tacotron2")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Patch PosixPath → WindowsPath globally before any torch import so that
# Linux-saved checkpoints (which pickle pathlib.PosixPath objects) can be
# loaded on Windows without UnsupportedOperation errors.
if not hasattr(pathlib, "PosixPath") or pathlib.PosixPath is not pathlib.WindowsPath:
    pathlib.PosixPath = pathlib.WindowsPath  # type: ignore[attr-defined]

import torch
import matplotlib.pyplot as plt
from IPython.display import Audio

import numpy as np
import soundfile as sf
import torchaudio

from commons.dataset import AudioMelConversions, denormalize, normalize
from commons.hyperparams import Tacotron2Config, WaveRNNConfig
from model import Tacotron2
from tokenizer import Tokenizer
from wavernn.wavernn import WaveRNN
from wavernn.hifigan import load_hifigan

_CKPT_CANDIDATES = [
    os.path.join(_REPO_ROOT, "checkpoints_omani", "B", "speaker_B_last.pth"),
    os.path.join(_REPO_ROOT, "speaker_B_last.pth"),
    os.path.join(_REPO_ROOT, "..", "speaker_B_last.pth"),
]
CHECKPOINT_PATH = next(
    (os.path.normpath(p) for p in _CKPT_CANDIDATES if os.path.isfile(p)), None
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRIFFIN_LIM_ITERS = 60

PITCH_SHIFT_SEMITONES = -5
EDGE_TRIM_MS = 50

wavernn_config = WaveRNNConfig()


def _build_wavernn(cfg: WaveRNNConfig) -> WaveRNN:
    product = 1
    for s in cfg.upsample_scales:
        product *= s
    assert product == cfg.hop_length
    return WaveRNN(
        upsample_scales=list(cfg.upsample_scales),
        n_classes=cfg.n_classes,
        hop_length=cfg.hop_length,
        n_res_block=cfg.n_res_block,
        n_rnn=cfg.n_rnn,
        n_fc=cfg.n_fc,
        kernel_size=cfg.kernel_size,
        n_freq=cfg.n_mels,
        n_hidden=cfg.n_hidden,
        n_output=cfg.n_output,
    )


# ── HiFi-GAN (Arabic-trained, hifigan-asc.pth) ───────────────────────────────
hifigan_model = None
_asc_ckpt_path = os.path.join(_REPO_ROOT, "hifigan-asc.pth")
if os.path.isfile(_asc_ckpt_path):
    try:
        hifigan_model = load_hifigan(_asc_ckpt_path, device=DEVICE)
        print(f"Loaded HiFi-GAN (Arabic) from {_asc_ckpt_path} on {DEVICE}")
    except Exception as _e:
        print(f"Failed to load hifigan-asc.pth ({_e}) — falling back.")
else:
    print(f"hifigan-asc.pth not found — trying WaveRNN / Griffin–Lim.")

# ── WaveRNN (fallback) ────────────────────────────────────────────────────────
wavernn_model = None
if hifigan_model is None:
    _WAVERNN_CKPT_DIR = os.path.join(_REPO_ROOT, "wavernn_checkpoints", "checkpoints")
    _WAVERNN_CKPT_CANDIDATES = [os.path.join(_WAVERNN_CKPT_DIR, "wavernn_last.pt")]
    if os.path.isdir(_WAVERNN_CKPT_DIR):
        _ep = sorted(glob.glob(os.path.join(_WAVERNN_CKPT_DIR, "wavernn_epoch*.pt")))
        _WAVERNN_CKPT_CANDIDATES.extend(reversed(_ep))
    WAVERNN_CHECKPOINT_PATH = next(
        (os.path.normpath(p) for p in _WAVERNN_CKPT_CANDIDATES if os.path.isfile(p)), None
    )
    if WAVERNN_CHECKPOINT_PATH is not None:
        try:
            wck = torch.load(WAVERNN_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
        except TypeError:
            wck = torch.load(WAVERNN_CHECKPOINT_PATH, map_location="cpu")
        wavernn_model = _build_wavernn(wavernn_config).to(DEVICE)
        wavernn_model.load_state_dict(wck["model"], strict=True)
        wavernn_model.eval()
        print(f"Loaded WaveRNN {WAVERNN_CHECKPOINT_PATH} on {DEVICE}")
    else:
        print("No WaveRNN found — using Griffin–Lim.")
# ─────────────────────────────────────────────────────────────────────────────


def _load_checkpoint(path: str, map_location=None):
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(120)
    if size < 20_000:
        if b"git-lfs.github.com" in head or b"version https://git-lfs" in head:
            raise RuntimeError("Git LFS pointer — run `git lfs pull`.")
        raise RuntimeError(f"Checkpoint only {size} bytes — too small.")
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        ck = torch.load(path, map_location="cpu")
    except RuntimeError as e:
        err = str(e).lower()
        if "zip" in err or "central directory" in err or "pytorchstreamreader" in err:
            raise RuntimeError(f"Truncated/corrupted checkpoint: {path}") from e
        raise
    if isinstance(ck, dict) and "model_state_dict" in ck:
        return ck.get("config"), ck["model_state_dict"]
    return None, ck


if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "No checkpoint found. Tried:\n"
        + "\n".join(f"  - {os.path.normpath(p)}" for p in _CKPT_CANDIDATES)
    )

saved_config, state_dict = _load_checkpoint(CHECKPOINT_PATH, DEVICE)
config = saved_config if saved_config is not None else Tacotron2Config()
model = Tacotron2(config).to(DEVICE)
model.load_state_dict(state_dict, strict=True)
model.eval()

tokenizer = Tokenizer()
a2m = AudioMelConversions(
    num_mels=config.num_mels,
    sampling_rate=config.sample_rate,
    n_fft=config.n_fft,
    window_size=config.win_length,
    hop_size=config.hop_length,
    fmin=config.fmin,
    fmax=config.fmax,
    min_db=config.min_db,
    max_scaled_abs=config.max_scaled_abs,
)
print(f"Loaded Tacotron2 from {CHECKPOINT_PATH} on {DEVICE}")


def _post_process(
    audio: np.ndarray,
    sample_rate: int,
    pitch_shift_semitones: float = 0.0,
    edge_trim_ms: float = 30.0,
) -> np.ndarray:
    idx = np.where(np.abs(audio) > 5e-3)[0]
    if idx.size > 0:
        audio = audio[idx[0]: idx[-1] + 1]

    trim = int(edge_trim_ms * sample_rate / 1000)
    trim = min(trim, len(audio) // 4)
    if trim > 0:
        audio = audio[trim: len(audio) - trim]

    if len(audio) == 0:
        return audio.astype(np.float32)

    audio = np.tanh(audio)

    fade = min(int(0.010 * sample_rate), len(audio) // 4)
    if fade > 0:
        audio[:fade]  *= np.linspace(0.0, 1.0, fade)
        audio[-fade:] *= np.linspace(1.0, 0.0, fade)

    if pitch_shift_semitones != 0.0:
        ratio = 2.0 ** (pitch_shift_semitones / 12.0)
        virtual_sr = int(round(sample_rate / ratio))
        t = torch.from_numpy(audio).unsqueeze(0)
        audio = torchaudio.functional.resample(
            t, orig_freq=virtual_sr, new_freq=sample_rate,
            resampling_method="sinc_interp_kaiser",
        ).squeeze(0).numpy()

    peak = float(np.max(np.abs(audio)))
    if peak > 0:
        audio = 0.95 * (audio / peak)

    return audio.astype(np.float32)


def inference(
    text,
    griffin_lim_iters=GRIFFIN_LIM_ITERS,
    max_decode_steps=2000,
    save_path=None,
    pitch_shift_semitones=PITCH_SHIFT_SEMITONES,
    edge_trim_ms=EDGE_TRIM_MS,
):
    print(f"Input text: {text!r}")
    tokens = tokenizer.encode(text).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        mel_post, alignments = model.inference(tokens, max_decode_steps=max_decode_steps)

    fig, axes = plt.subplots(2, 1, figsize=(6, 7))
    im0 = axes[0].imshow(
        mel_post[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[0].set_title("Predicted mel (post-net)")
    axes[0].set_ylabel("Mel bin")
    fig.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(
        alignments[0].T.cpu().numpy(), aspect="auto", origin="lower", interpolation="none"
    )
    axes[1].set_title("Attention alignment")
    axes[1].set_xlabel("Decoder time")
    axes[1].set_ylabel("Encoder position")
    fig.colorbar(im1, ax=axes[1])
    plt.tight_layout()
    plt.show()

    if hifigan_model is not None:
        print("Vocoder: HiFi-GAN (Arabic)")
        mel_db = denormalize(mel_post[0].T.float().cpu(), min_db=config.min_db, max_abs_val=config.max_scaled_abs)
        mel_ln = mel_db * (math.log(10) / 20)
        with torch.inference_mode():
            wav = hifigan_model.infer(mel_ln.unsqueeze(0).to(DEVICE))
        audio_f32 = wav.squeeze().cpu().numpy().astype(np.float32)

    elif wavernn_model is not None:
        print("Vocoder: WaveRNN")
        mel_tac = mel_post[0].T.float().cpu()
        mel_db = denormalize(mel_tac, min_db=config.min_db, max_abs_val=config.max_scaled_abs)
        mel_wr = normalize(mel_db, min_db=wavernn_config.min_db, max_abs_val=wavernn_config.max_scaled_abs).to(DEVICE)
        with torch.inference_mode():
            wav, _ = wavernn_model.infer(mel_wr.unsqueeze(0))
        audio_f32 = wav[0, 0].float().cpu().numpy()

    else:
        print("Vocoder: Griffin–Lim")
        audio_i16 = a2m.mel2audio(
            mel_post[0].T.cpu(), do_denorm=True, griffin_lim_iters=griffin_lim_iters
        )
        audio_f32 = audio_i16.astype(np.float32) / 32768.0

    audio_f32 = _post_process(audio_f32, config.sample_rate, pitch_shift_semitones, edge_trim_ms)

    if save_path:
        save_path = os.path.normpath(os.path.join(_REPO_ROOT, save_path))
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        sf.write(save_path, audio_f32, config.sample_rate)
        print(f"Saved audio to: {save_path}")

    display(Audio(audio_f32, rate=config.sample_rate))
    return mel_post, alignments, audio_f32

In [ ]:
inference(
    "ٱلسَّلَامُ عَلَيْكُمْ وَرَحْمَةُ ٱللّٰهِ وَبَرَكَاتُهُ",
    save_path=os.path.join("outputs", "speaker_B_last_sample.wav"),
)


In [ ]:

test_texts = [
    "ٱلسَّلَامُ عَلَيْكُمْ وَرَحْمَةُ ٱللّٰهِ وَبَرَكَاتُهُ",
    "بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ",
    "كَيْفَ حَالُكَ يَا صَدِيقِي؟",
    "أَهْلًا وَسَهْلًا بِكُمْ جَمِيعًا",
    "إِنَّ اللَّهَ عَلَى كُلِّ شَيْءٍ قَدِيرٌ",
    "مَرْحَبًا بِكَ فِي بِلَادِنَا الْجَمِيلَةِ",
    "الْعِلْمُ نُورٌ وَالْجَهْلُ ظَلَامٌ",
    "صَبَاحُ الْخَيْرِ يَا أَصْدِقَائِي",
]

for i, text in enumerate(test_texts):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}/{len(test_texts)}")
    inference(
        text,
        save_path=os.path.join("outputs", f"test_sample_{i+1:02d}.wav"),
    )


In [ ]:
# Diagnostic: run WaveRNN on a ground-truth mel at both normalization scales.
# Uses the same row loading as WaveRNNDataset (resamples to wavernn_config.sample_rate).
import pandas as pd
from commons.hyperparams import WAVERNN_VAL_PATH

val_df = pd.read_parquet(WAVERNN_VAL_PATH)
row = val_df.iloc[0]
audio_raw = row["audio"]
if isinstance(audio_raw, (bytes, bytearray)):
    audio_raw = np.frombuffer(audio_raw, dtype=np.float32).copy()
audio_gt = torch.tensor(np.asarray(audio_raw), dtype=torch.float32).reshape(-1)

# Match WaveRNNDataset._row_audio: if the row has an 'sr' field, resample to target.
orig_sr = int(row["sr"]) if "sr" in row.index else wavernn_config.sample_rate
if orig_sr != wavernn_config.sample_rate:
    audio_gt = torchaudio.functional.resample(
        audio_gt, orig_freq=orig_sr, new_freq=wavernn_config.sample_rate
    )
print(f"GT audio: orig_sr={orig_sr} -> {wavernn_config.sample_rate}, "
      f"{audio_gt.numel()} samples ({audio_gt.numel()/wavernn_config.sample_rate:.2f}s)")

sf.write(os.path.join(_REPO_ROOT, "outputs", "gt_reference.wav"),
         audio_gt.numpy(), wavernn_config.sample_rate)
print("--- Ground truth reference (should sound natural) ---")
display(Audio(audio_gt.numpy(), rate=wavernn_config.sample_rate))

for scale in (1.0, 4.0):
    a2m_test = AudioMelConversions(
        num_mels=wavernn_config.n_mels,
        sampling_rate=wavernn_config.sample_rate,
        n_fft=wavernn_config.n_fft,
        window_size=wavernn_config.window_size,
        hop_size=wavernn_config.hop_length,
        fmin=wavernn_config.fmin,
        fmax=wavernn_config.fmax,
        center=False,
        min_db=wavernn_config.min_db,
        max_scaled_abs=scale,
    )
    mel_gt = a2m_test.audio2mel(audio_gt, do_norm=True).to(DEVICE)
    print(f"\n--- max_scaled_abs={scale} ---")
    print(f"mel range: min={mel_gt.min().item():.3f} max={mel_gt.max().item():.3f} "
          f"shape={tuple(mel_gt.shape)}")
    with torch.inference_mode():
        wav, _ = wavernn_model.infer(mel_gt.unsqueeze(0))
    w = wav[0, 0].float().cpu().numpy()
    print(f"wav peak: {np.max(np.abs(w)):.4f}  mean|w|: {np.mean(np.abs(w)):.4f}")
    out_path = os.path.join(_REPO_ROOT, "outputs", f"gt_mel_scale{scale}.wav")
    sf.write(out_path, w, wavernn_config.sample_rate)
    display(Audio(w, rate=wavernn_config.sample_rate))
